In [2]:
import os
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
import numpy as np

base_dir = Path('E:/hneu/data')
output_dir = Path('E:/hneu/processed_data')
output_dir.mkdir(exist_ok=True)

category_mapping = {
    'fwdfinalconsumer': 'Consumer',
    'fwdfinalenergyandultilities': 'EnergyAndUtilities',
    'fwdfinalhealthcare': 'Healthcare',
    'fwdfinalindustrials': 'Industrials',
    'fwdfinalmaterials': 'Materials',
    'fwdfinalrealestate': 'RealEstate',
    'fwdfinaltelecompart1': 'Telecom',
    'fwdfinaltelecompart2': 'Telecom'
}

def clean_numeric_value(x):
    if pd.isna(x) or x == '' or x == 'NA' or x == 'NM':
        return np.nan
    if isinstance(x, str):
        x = x.strip()
        if x == 'NA' or x == 'NM':
            return np.nan
        if x.startswith('(') and x.endswith(')'):
            x = '-' + x[1:-1]
        x = x.replace(',', '').replace(' ', '')
        try:
            return float(x)
        except:
            return np.nan
    return x

for category_folder in base_dir.iterdir():
    if not category_folder.is_dir() or 'fwdfinal' not in category_folder.name:
        continue
    
    category_name = category_mapping.get(category_folder.name, 'Unknown')
    category_output = output_dir / category_name
    category_output.mkdir(exist_ok=True)
    
    for excel_file in category_folder.glob('*.xlsx'):
        country_name = excel_file.stem
        country_output = category_output / country_name
        country_output.mkdir(exist_ok=True)
        
        try:
            df = pd.read_excel(excel_file, header=None)
            
            if len(df) < 6:
                continue
                
            col_names = df.iloc[2].values
            field_codes = df.iloc[3].values
            periods = df.iloc[4].values
            
            columns = []
            for i in range(len(col_names)):
                col_name = str(col_names[i]).strip() if not pd.isna(col_names[i]) else f'col_{i}'
                period = str(periods[i]).strip() if not pd.isna(periods[i]) else 'Unknown'
                if col_name == 'nan' or col_name == '':
                    col_name = f'col_{i}'
                if period == 'nan' or period == '':
                    period = 'Unknown'
                columns.append(f'{col_name}_{period}')
            
            data_df = df.iloc[5:].copy()
            data_df.columns = columns
            
            for col in data_df.columns:
                data_df[col] = data_df[col].apply(clean_numeric_value)
            
            data_df = data_df[~data_df.apply(lambda row: row.astype(str).isin(["NA", 'NA', "N/A", "na"]).any(), axis=1)]
            data_df = data_df.dropna(how='all')
            
            if not data_df.empty:
                json_path = country_output / f'{country_name}_{category_name}.json'
                parquet_path = country_output / f'{country_name}_{category_name}.parquet'
                
                data_df.to_json(json_path, orient='records', indent=2)
                
                table = pa.Table.from_pandas(data_df)
                pq.write_table(table, parquet_path)
                
        except Exception as e:
            print(f'Error processing {excel_file}: {str(e)}')

Error processing E:\hneu\data\fwdfinalconsumer\~$Argentina.xlsx: [Errno 13] Permission denied: 'E:\\hneu\\data\\fwdfinalconsumer\\~$Argentina.xlsx'
